# mixup: Beyond Empirical Risk Minimization — ICLR 2018 Reproduction
**Student Roll No**: 24CS3055  
**Department**: Department of Computer Science & Engineering, RGIPT  
**Paper ID**: DL-008 (ICLR 2018)  
**Target Hardware**: Free-tier Google Colab (Nvidia T4 GPU)

## 1. Environment & Determinism Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import random
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print("GPU Model:", torch.cuda.get_device_name(0))

## 2. Load Package Modules from `src/`

In [ ]:
from src.mixup import mixup_data, mixup_criterion
from src.models import PreActResNet18
from src.noise import add_label_noise
from src.calibration import compute_ece, plot_reliability_diagram
from train import train_one_epoch, evaluate

## 3. CIFAR-10 Dataset & Preprocessing

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform_train)
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform_test)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

## 4. Run Benchmark Experiments & Automatic Matplotlib Plotting

In [ ]:
# Execute Clean CIFAR-10 ERM vs Mixup Benchmark
epochs = 40
print("=== Training Vanilla ERM (Alpha = 0.0) ===")
m_erm = PreActResNet18().to(device)
opt_erm = torch.optim.SGD(m_erm.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
sched_erm = torch.optim.lr_scheduler.CosineAnnealingLR(opt_erm, T_max=epochs)
crit = nn.CrossEntropyLoss()

erm_errs = []
for ep in range(1, epochs + 1):
    train_one_epoch(m_erm, trainloader, opt_erm, crit, alpha=0.0, device=device)
    sched_erm.step()
    val_loss, val_acc, _, _ = evaluate(m_erm, testloader, crit, device)
    erm_errs.append(100.0 - val_acc)

print("\n=== Training Mixup (Alpha = 1.0) ===")
m_mix = PreActResNet18().to(device)
opt_mix = torch.optim.SGD(m_mix.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
sched_mix = torch.optim.lr_scheduler.CosineAnnealingLR(opt_mix, T_max=epochs)

mix_errs = []
for ep in range(1, epochs + 1):
    train_one_epoch(m_mix, trainloader, opt_mix, crit, alpha=1.0, device=device)
    sched_mix.step()
    val_loss, val_acc, _, _ = evaluate(m_mix, testloader, crit, device)
    mix_errs.append(100.0 - val_acc)

# Automatic Matplotlib Plot: Test Error Evolution (Paper Fig 3b)
plt.figure(figsize=(7, 5))
plt.plot(erm_errs, label='ERM (alpha=0.0)', color='orange', linewidth=2)
plt.plot(mix_errs, label='Mixup (alpha=1.0)', color='blue', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Top-1 Test Error (%)')
plt.title('Figure 3b Reproduction: Test Error Evolution')
plt.legend()
plt.grid(True, linestyle=':')
plt.savefig('../results/figures/fig3b_reproduction.png', dpi=300)
plt.show()